In [2]:
import pandas as pd
import matplotlib.pyplot as plt 

In [22]:
import pandas as pd
import dash
from dash import dcc, html, dash_table, Input, Output
import plotly.express as px

df = pd.read_csv("mereja.csv")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Title"] = df["Title"].fillna("")
df["Author"] = df["Author"].fillna("Unknown")
df["URL"] = df["URL"].fillna("")
df["Detail"] = df["Detail"].fillna("")

df = df.dropna(subset=["Date"]).copy()

df["DateOnly"] = df["Date"].dt.date

author_options = [{"label": "All Authors", "value": "All"}] + [
    {"label": author, "value": author}
    for author in sorted(df["Author"].unique())
]

app = dash.Dash(__name__)
app.title = "Article Dashboard"

card_style = {
    "flex": "1",
    "padding": "20px",
    "borderRadius": "10px",
    "backgroundColor": "white",
    "boxShadow": "0 2px 6px rgba(0,0,0,0.1)",
    "textAlign": "center"
}

app.layout = html.Div([
    html.H1("Mereja News", style={"textAlign": "center"}),

    html.Div([
        html.Div([
            html.H3("Filters"),
            html.Label("Search by Title or Detail"),
            dcc.Input(
                id="search-input",
                type="text",
                placeholder="Search article...",
                style={"width": "100%", "marginBottom": "15px"}
            ),

            html.Label("Filter by Author"),
            dcc.Dropdown(
                id="author-dropdown",
                options=author_options,
                value="All",
                clearable=False,
                style={"marginBottom": "15px"}
            ),

            html.Label("Filter by Date Range"),
            dcc.DatePickerRange(
                id="date-range",
                min_date_allowed=df["Date"].min().date(),
                max_date_allowed=df["Date"].max().date(),
                start_date=df["Date"].min().date(),
                end_date=df["Date"].max().date(),
                display_format="YYYY-MM-DD"
            ),
        ], ),

        html.Div([
            html.Div([
                html.Div([
                    html.H4("Total Articles"),
                    html.H2(id="total-articles")
                ], style=card_style),

                html.Div([
                    html.H4("Unique Authors"),
                    html.H2(id="unique-authors")
                ], style=card_style),

                html.Div([
                    html.H4("Filtered Articles"),
                    html.H2(id="filtered-articles")
                ], style=card_style),
            ], style={"display": "flex", "gap": "20px", "marginBottom": "20px"}),

            dcc.Graph(id="articles-trend-chart"),
            dcc.Graph(id="author-bar-chart"),
            html.H3("Articles Table"),

            dash_table.DataTable(
                id="articles-table",
                columns=[
                    {"name": "Title", "id": "Title"},
                    {"name": "Author", "id": "Author"},
                    {"name": "Date", "id": "Date"},
                    {"name": "URL", "id": "URL", "presentation": "markdown"},
                    {"name": "Detail", "id": "Detail"},
                ],
                data=[],
                page_size=10,
                style_table={"overflowX": "auto"},
                style_cell={
                    "textAlign": "left",
                    "padding": "8px",
                    "whiteSpace": "normal",
                    "height": "auto",
                    "maxWidth": "250px"
                },
                style_header={
                    "backgroundColor": "#343a40",
                    "color": "white",
                    "fontWeight": "bold"
                },
                row_selectable="single",
                selected_rows=[]
            ),

            html.Br(),

            html.H3("Selected Article Detail"),
            html.Div(
                id="article-detail-box",
                style={
                    "padding": "15px",
                    "border": "1px solid #ddd",
                    "borderRadius": "8px",
                    "backgroundColor": "#fafafa",
                    "minHeight": "120px"
                }
            )
        ], style={
            "width": "75%",
            "display": "inline-block",
            "verticalAlign": "top",
            "padding": "20px"
        })
    ])
], style={"fontFamily": "Arial, sans-serif", "padding": "20px"})


@app.callback(
    Output("total-articles", "children"),
    Output("unique-authors", "children"),
    Output("filtered-articles", "children"),
     Output("articles-trend-chart", "figure"),
    Output("author-bar-chart", "figure"),
    Output("articles-table", "data"),
    Input("search-input", "value"),
    Input("author-dropdown", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date")
)
def update_dashboard(search_text,selected_author, start_date, end_date):
    filtered_df = df.copy()

    if selected_author != "All":
        filtered_df = filtered_df[filtered_df["Author"] == selected_author]

    if start_date and end_date:
        filtered_df = filtered_df[
            (filtered_df["Date"] >= pd.to_datetime(start_date)) &
            (filtered_df["Date"] <= pd.to_datetime(end_date))
        ]
    if search_text and search_text.strip():
        search_text = search_text.lower()
        filtered_df = filtered_df[
            filtered_df["Title"].str.lower().str.contains(search_text, na=False) |
            filtered_df["Detail"].str.lower().str.contains(search_text, na=False)
        ]
    trend_df = filtered_df.groupby("DateOnly").size().reset_index(name="Article Count")
    trend_fig = px.line(
        trend_df,
        x="DateOnly",
        y="Article Count",
        markers=True,
        title="Articles Published Over Time"
    )
    trend_fig.update_layout(template="plotly_white")

    author_df = filtered_df["Author"].value_counts().reset_index()
    author_df.columns = ["Author", "Article Count"]

    author_fig = px.bar(
        author_df,
        x="Author",
        y="Article Count",
        title="Articles by Author"
    )
    author_fig.update_layout(template="plotly_white", xaxis_tickangle=-45)


    total_articles = len(df)
    unique_authors = filtered_df["Author"].nunique()
    filtered_articles = len(filtered_df)

    table_df = filtered_df.copy()
    table_df["Date"] = table_df["Date"].dt.strftime("%Y-%m-%d")
    table_df["URL"] = table_df["URL"].apply(lambda x: f"[Open Link]({x})" if x else "")

    return (
        total_articles,
        unique_authors,
        filtered_articles,
        trend_fig,
        author_fig,
        table_df[["Title", "Author", "Date", "URL", "Detail"]].to_dict("records")
    )

@app.callback(
    Output("article-detail-box", "children"),
    Input("articles-table", "derived_virtual_data"),
    Input("articles-table", "selected_rows")
)
def show_article_detail(rows, selected_rows):
    if not rows or not selected_rows:
        return "Select a row from the table to view article details."

    row = rows[selected_rows[0]]

    url = row["URL"]
    if url.startswith("[Open Link](") and url.endswith(")"):
        url = url[len("[Open Link]("):-1]

    return html.Div([
        html.H4(row["Title"]),
        html.P([html.B("Author: "), row["Author"]]),
        html.P([html.B("Date: "), row["Date"]]),
        html.P([
            html.B("URL: "),
            html.A("Open Article", href=url, target="_blank") if url else "No URL"
        ]),
        html.Hr(),
        html.P(row["Detail"])
    ])


if __name__ == "__main__":
    app.run(debug=True)